# Group 10 — Combined vs Two-phase M1 (Colab CPU-only)

Notebook thử nghiệm riêng, không sửa notebook Pilot. Chưa có kết quả Colab.
Mỗi phương án chạy cùng **50 ID ứng viên** từ bốn khoảng Group 10, seed cố định,
không lọc theo tồn tại. Mọi version và revision history được bật.
Đọc [hướng dẫn](../docs/colab_comparison.md) trước khi bấm nút gọi API.
Chọn Runtime → Change runtime type → None. Chỉ một writer, không chạy A/B đồng thời.
Chạy các ô setup chỉ chuẩn bị giao diện; không tự crawl hoặc kiểm tra S2.

In [ ]:
# Full hash của commit đã review có đủ notebook/helper/dependencies mới.
CODE_COMMIT = ""
EXPERIMENT_ID = "m1-ab-50-01"  # mới cho trial lạnh; giữ nguyên khi Resume
SEED = 10
REPOSITORY_URL = "https://github.com/Toan-ten/Arxiv-Project-Group10.git"
# Hai arms dùng nguyên cấu hình này; thay đổi cấu hình cần experiment mới.
S2_CONFIG = dict(timeout=30, max_attempts=1, min_interval=2,
                 reference_limit=100, max_pages=1000)

In [ ]:
from pathlib import Path
import importlib.metadata
import json
import os
import re
import subprocess
import sys
from google.colab import drive, output, userdata

if sys.version_info < (3, 10) or not re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT):
    raise ValueError("Cần Python >=3.10 và full commit đã review")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", EXPERIMENT_ID):
    raise ValueError("EXPERIMENT_ID không hợp lệ")
drive.mount("/content/drive")
if not os.path.ismount("/content/drive"):
    raise RuntimeError("Drive chưa mount")
REPO = Path("/content/Arxiv-Project-Group10")
if not REPO.exists():
    subprocess.run(["git", "clone", "--single-branch", "--branch", "feat/m1-integration",
                    REPOSITORY_URL, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_COMMIT], check=True)
def git(*args):
    return subprocess.check_output(["git", "-C", str(REPO), *args], text=True).strip()
if git("rev-parse", "HEAD") != CODE_COMMIT or git("status", "--porcelain"):
    raise RuntimeError("Clone không đúng commit/sạch; không tự reset hoặc cập nhật")
if git("remote", "get-url", "origin") != REPOSITORY_URL:
    raise RuntimeError("Sai origin")
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                str(REPO / "requirements-colab-compare.txt")], check=True)
import psutil
if psutil.__version__ != importlib.metadata.version("psutil"):
    raise RuntimeError("psutil đã cache phiên bản khác: restart session rồi chạy setup lại")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
output.enable_custom_widget_manager()
print("Pinned code:", CODE_COMMIT, "Python:", sys.version.split()[0])

## Chuẩn bị filesystem và giao diện

Corpora/logs/reports A và B tách biệt trên Drive. Scratch tách A/B trên `/content`.
Fixture preflight dưới đây kiểm tra I/O, replace và SHA-256, không chứng minh
crash durability của Drive. S2 key chỉ được đọc từ Colab Secrets khi bấm nút mạng;
không ghi .env/key hoặc cache metadata preflight vào hai corpus.

Bấm **S2 preflight (4 IDs)** trước 50 ID: tối đa một POST metadata (attempts=1),
không tải references hoặc source. Thành công chỉ chứng minh request đó hợp lệ,
không bảo đảm 50 ID tiếp theo không gặp 429.

In [ ]:
import ipywidgets as widgets
from IPython.display import HTML, display
from scripts.colab_comparison import Comparison, build_controls, filesystem_probe

ROOT = Path("/content/drive/MyDrive/Group10-M1-Comparison") / EXPERIMENT_ID
SCRATCH = Path("/content/group10-comparison-scratch") / EXPERIMENT_ID
session = Comparison(ROOT, SCRATCH, CODE_COMMIT, seed=SEED, s2_config=S2_CONFIG,
    secret_loader=lambda: userdata.get("SEMANTIC_SCHOLAR_API_KEY"))
probe = filesystem_probe(ROOT / "filesystem-preflight")
print("Filesystem fixture:", probe['verified'], probe['probe_path'])
print("Selection fingerprint:", session.definition['selection_fingerprint'])
print("50 candidate IDs:", session.pairs)
controls = build_controls(session, widgets, display, lambda html: display(HTML(html)))

## Thao tác bằng nút

1. Lưu key trong Colab Secrets tên `SEMANTIC_SCHOLAR_API_KEY`, cấp quyền notebook.
2. Tick CPU-only và Only writer sau khi xác nhận; bấm S2 preflight, xem status/counts.
3. Bấm Run A hoặc Run B, chờ kết thúc rồi mới chạy phương án kia. Không có nút Run all.
   B tự chạy arXiv toàn cohort rồi S2 bằng batch ID khác trên cùng corpus nếu dữ liệu
   arXiv đủ hợp lệ. Nếu dừng/lỗi, không chạy tiếp pha hoặc phương án khác tự động.
4. Sau 403/429: dừng truy cập, xem logs/cooldown/quota; chỉ tick ô Policy stop reviewed
   sau khi thực sự review. Mỗi nút mạng là một quyết định mới, không có retry loop.
5. Resume A/B dùng cùng experiment/config/commit, nút riêng. Runtime mới: mount/setup
   lại, giữ EXPERIMENT_ID, bỏ nút Run mới. Dữ liệu hỏng phải audit, không xóa để ép chạy.
6. Bấm Audit + compare để lưu JSON/CSV mới và xem bảng. Đây là đọc corpus/logs, không HTTP.

`cold_end_to_end_seconds` đo toàn hành động nút bấm, gồm chờ và kiểm tra giữa hai pha.
`cold_wall_seconds` là riêng callback crawler lượt đầu; `active_wall_seconds` cộng mọi
attempts duy nhất. Không tính khoảng nghỉ giữa phiên. RAM average theo số samples; peak lấy max;
disk snapshot cuối không cộng hai pha. Kernel notebook nằm trong RSS; disk chỉ gồm
corpus+scratch của phương án, không gồm corpus phương án kia, Git/logs/reports.
Preflight nằm ngoài A/B. Boundary waits/phase gates nằm trong end-to-end, ngoài
callback wall. Resource sampling phủ callback crawler, không đo liên tục mọi audit/report.

Nếu stopped, thiếu phép đo/interval, kết quả chưa đầy đủ/khác nhau hoặc runtime đổi:
bảng phải ghi **không đủ bằng chứng kết luận tốc độ**. Một trial 50 ID không đại diện
20.000 ID. Thử trial mới cùng seed theo thứ tự B→A để kiểm tra hiệu ứng thứ tự/quota;
không chạy song song hoặc dùng nhiều runtime để tăng request rate.